# Sesión 3 · Ejercicio 1 — Difusión desde cero en 2D
**Formato:** EN PLENARIA, pantalla compartida. La instructora escribe
cada bloque; usted lo replica y lo ejecuta. **Corre en CPU.**
**Objetivo:** construir un modelo de difusión completo en ~40 líneas y
*ver* el ruido organizarse en datos.
**Tiempo:** 45 min
**Produce:** §5 de su bitácora — la figura de la trayectoria


### Cómo trabajar este cuaderno (1 minuto de lectura)

1. **Guarde su copia**: Archivo → Guardar una copia en Drive. Si no, pierde su trabajo al cerrar.
2. Ejecute las celdas **en orden**. Solo las marcadas `#### OBLIGATORIO ####` producen su entregable; las de **EXTENSIÓN** son opcionales, para quien le sobre tiempo.
3. ¿Algo no corre, o tarda demasiado? Ejecute la **CELDA DE RESCATE**: carga resultados ya calculados y usted sigue con el análisis. Usarla **no descuenta puntos** — solo dígalo en su bitácora.
4. Al terminar, copie la figura y sus observaciones (2-3 líneas con sus palabras) a la sección de su **bitácora** que dice el encabezado. Eso es TODO el entregable — no se pide nada más.


In [ ]:
#### OBLIGATORIO #### — setup (idempotente: puede ejecutarla dos veces)
import os, sys
if not os.path.isdir("src"):
    if not os.path.isdir("IAA6_M13_Gen"):
        !git clone -q https://github.com/AdriannaGmz/IAA6_M13_Gen
    %cd IAA6_M13_Gen
# Si ya había una copia en esta máquina, se pone al día: de lo
# contrario se quedaría con la versión del día que la clonó, y las
# correcciones publicadas después nunca le llegarían.
!git pull -q --ff-only
!pip install -q -r requirements.txt
sys.path.insert(0, ".")
from src import datos, modelos, evaluar, graficas, rescate
MODO_GPU = rescate.hay_gpu()   # imprime "GPU disponible" o "Modo CPU"


In [ ]:
#### OBLIGATORIO #### — Bloque 2: el conjunto 2D (dos lunas)
# Se construye EN VIVO: replique lo de la pantalla. El esqueleto
# queda aquí por si pierde el hilo.
# COMPLETAR: deje X2 centrado y con escala unitaria.
# Pista: réstele la media por columna y divida entre la desviación.
import torch
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
X2 = torch.tensor(make_moons(2000, noise=0.05, random_state=0)[0],
                  dtype=torch.float32)
X2 = ...
plt.scatter(X2[:, 0], X2[:, 1], s=4); plt.axis("equal"); plt.show()


### La idea completa en dos renglones
1. **Hacia adelante (diseñado, sin aprendizaje):** destruir los datos
   añadiendo ruido poco a poco, en T pasos, hasta que sólo quede ruido.
2. **Hacia atrás (aprendido):** una red que en cada paso predice el
   ruido añadido, para poder quitarlo. Generar = empezar desde ruido
   puro y quitar ruido T veces.


In [ ]:
#### OBLIGATORIO #### — Bloque 3: el calendario de ruido (parte DISEÑADA)
# Se construye EN VIVO: replique lo de la pantalla. El esqueleto
# queda aquí por si pierde el hilo.
# COMPLETAR: cuánto ruido entra por paso, y cuánta señal queda al paso t.
# Pista 1: betas son T valores que suben de 1e-4 a 0.02 (torch.linspace).
# Pista 2: alfas_cum es el producto ACUMULADO de (1 - betas)
#          (torch.cumprod). Nada de esto se aprende: es diseñado.
T = 200
betas = ...
alfas_cum = ...
print(f"proporción de señal restante:  t=0 → {alfas_cum[0]:.3f}   "
      f"t=100 → {alfas_cum[100]:.3f}   t=199 → {alfas_cum[-1]:.4f}")


**Bloque 4 — ver la destrucción.** La función `ensuciar` es la fórmula
del «salto directo»: del dato limpio al paso *t* sin recorrer los
pasos intermedios. Ojo con el título de la figura: dice «proceso
inverso» porque reutilizamos la función de gráficas, pero aquí vamos
en dirección de DESTRUCCIÓN (la dirección buena llega en el Bloque 9).


In [ ]:
#### OBLIGATORIO #### — Bloque 4: destruir con ruido, en cuatro instantes
# Se construye EN VIVO: replique lo de la pantalla. El esqueleto
# queda aquí por si pierde el hilo.
# COMPLETAR: la mezcla de dato y ruido en el paso t.
# Pista: se conserva sqrt(ac) del dato y se añade sqrt(1 - ac) de
#        ruido gaussiano (torch.randn_like). Un solo renglón.
def ensuciar(x0, t):
    ac = alfas_cum[t]
    return ...

instantes = [0, 50, 100, 199]
graficas.trayectoria_2d([ensuciar(X2, t) for t in instantes],
                        titulos=[f"t = {t}" for t in instantes])


**Bloques 5–7 — la única parte aprendida.** Una red mínima (entra el
punto ruidoso + el número de paso; sale el ruido que cree que se
añadió), el ciclo de entrenamiento (que es la deducción de la clase
hecha código: sortear paso, sortear ruido, ensuciar, predecir, error
cuadrático) y ~1 minuto de CPU. La pérdida baja de ~0.93 a ~0.47 y se
**estanca — eso es lo correcto**: le pedimos predecir ruido aleatorio
y hay una parte que nadie puede adivinar.


In [ ]:
#### OBLIGATORIO #### — Bloque 5: la red quita-ruido (parte APRENDIDA)
# Se construye EN VIVO: replique lo de la pantalla. El esqueleto
# queda aquí por si pierde el hilo.
# COMPLETAR: la red.
# Entrada: (x, y, t/T) — el punto ruidoso y EN QUÉ paso estamos: 3.
# Salida: el ruido que la red cree que se añadió: 2. Nada más.
# Pista: nn.Sequential con dos capas ocultas de 128 y ReLU entre ellas.
import torch.nn as nn
import torch.nn.functional as F
red = ...


In [ ]:
#### OBLIGATORIO #### — Bloque 6: el ciclo — predecir el ruido añadido
# Se construye EN VIVO: replique lo de la pantalla. El esqueleto
# queda aquí por si pierde el hilo.
# COMPLETAR: el punto ruidoso del lote y la pérdida.
# Pista 1: x_t se arma igual que en el Bloque 4, pero con el `ruido`
#          y el `ac` que ya están calculados aquí.
# Pista 2: la red recibe torch.cat([x_t, t[:, None] / T], 1) y su
#          objetivo es el `ruido` REAL. Compare con F.mse_loss.
opt = torch.optim.Adam(red.parameters(), lr=1e-3)
def paso_entrenamiento():
    t = torch.randint(0, T, (len(X2),))
    ruido, ac = torch.randn_like(X2), alfas_cum[t][:, None]
    x_t = ...
    perdida = ...
    opt.zero_grad(); perdida.backward(); opt.step()
    return float(perdida)


In [ ]:
#### OBLIGATORIO #### — Bloque 7: entrenar (~1 min en CPU)
# Se construye EN VIVO: replique lo de la pantalla. El esqueleto
# queda aquí por si pierde el hilo.
# Este bloque no tiene nada que completar: es el ciclo de siempre.
torch.manual_seed(0)
for paso in range(3000):
    p = paso_entrenamiento()
    if paso % 500 == 0:
        print(f"  paso {paso:4d} · pérdida {p:.3f}")


**Bloques 8–9 — el camino de regreso.** El único bloque denso del día:
desde ruido puro, 200 iteraciones hacia atrás — en cada una la red
predice el ruido, se estima el dato limpio y se re-ensucia un poco
menos. Léalo dos veces antes de ejecutar. El Bloque 9 es el pago de
todo lo anterior: 1500 puntos de ruido organizándose en las dos lunas.


In [ ]:
#### OBLIGATORIO #### — Bloque 8: el ciclo de muestreo inverso
# Se construye EN VIVO: replique lo de la pantalla. El esqueleto
# queda aquí por si pierde el hilo.
# COMPLETAR: estimar el dato limpio y dar el paso hacia atrás.
# Pista 1: x0 se despeja de la fórmula del Bloque 4: al punto ruidoso
#          se le quita el ruido predicho y se divide entre sqrt(ac).
# Pista 2: para volver a ensuciar hasta t-1 se usa ac_prev con la
#          misma receta del Bloque 4, pero con `pred` en lugar de
#          ruido nuevo. El .clamp(-3, 3) evita que se dispare.
@torch.no_grad()
def muestrear(n, guardar_en=()):
    x, estados = torch.randn(n, 2), []
    for t in reversed(range(T)):
        pred = red(torch.cat([x, torch.full((n, 1), t / T)], 1))
        ac_prev = alfas_cum[t - 1] if t else torch.tensor(1.0)
        x0 = ...
        x = ...
        if t in guardar_en: estados.append(x.clone())
    return x, estados


In [ ]:
#### OBLIGATORIO #### — Bloque 9: ★ ver el ruido organizarse
# Se construye EN VIVO: replique lo de la pantalla. El esqueleto
# queda aquí por si pierde el hilo.
# Nada que completar: éste es el pago de todo lo anterior. La figura
# que guarda aquí es la que va a su §5.
torch.manual_seed(1)
muestras, estados = muestrear(1500, guardar_en={199, 150, 100, 50, 0})
fig = graficas.trayectoria_2d(
    estados, titulos=["t = 199", "t = 150", "t = 100", "t = 50",
                      "t = 0 (datos)"])
fig.savefig("bitacora_s3e1_trayectoria.png", dpi=120)


In [ ]:
# ── CELDA DE RESCATE ────────────────────────────────────────
# ¿Se perdió en la replicación o no entrenó? Ejecute esto y siga:
# deja el calendario, la red YA ENTRENADA, la función de muestreo y
# la figura de su §5 guardada en disco.
import torch, torch.nn as nn
T = 200
betas = torch.linspace(1e-4, 0.02, T)
alfas_cum = torch.cumprod(1 - betas, dim=0)
red = nn.Sequential(nn.Linear(3, 128), nn.ReLU(),
                    nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))
contenido, figuras = rescate.cargar("s3_ddpm2d")
red.load_state_dict(contenido["state_dict"])


def ensuciar(x0, t):
    ac = alfas_cum[t]
    return ac.sqrt() * x0 + (1 - ac).sqrt() * torch.randn_like(x0)


@torch.no_grad()
def muestrear(n, guardar_en=()):
    x, estados = torch.randn(n, 2), []
    for t in reversed(range(T)):
        pred = red(torch.cat([x, torch.full((n, 1), t / T)], 1))
        ac_prev = alfas_cum[t - 1] if t else torch.tensor(1.0)
        x0 = (x - (1 - alfas_cum[t]).sqrt() * pred) / alfas_cum[t].sqrt()
        x = ac_prev.sqrt() * x0.clamp(-3, 3) + (1 - ac_prev).sqrt() * pred
        if t in guardar_en: estados.append(x.clone())
    return x, estados


# La figura precomputada queda con el nombre que pide la bitácora,
# para que pueda entregar su §5 aunque el Bloque 9 no haya corrido.
figuras[0].savefig("bitacora_s3e1_trayectoria.png", dpi=120)
print("Listo: red entrenada, muestrear() disponible y")
print("bitacora_s3e1_trayectoria.png guardada.")


### Observación — complete antes de cerrar

- La red del Bloque 5 nunca ve el conjunto completo, sólo pares
  (punto ruidoso, ruido añadido). ¿En qué parte del código quedó
  «guardada» la forma de las dos lunas después de entrenar? ___
- ¿En qué instante de la trayectoria «aparece» la estructura? ___


In [ ]:
#### OBLIGATORIO #### — artefacto para la bitácora
print("Copie este bloque en la sección §5 de su bitácora y adjunte")
print("bitacora_s3e1_trayectoria.png:\n")
print("- T =", T, "pasos · red de 3 capas · objetivo: predecir el ruido")
print("- ¿Dónde quedó guardada la forma de los datos?: <...>")
print("- ¿Cuándo aparece la estructura?: <...>")


### EXTENSIÓN (si termina antes)
Cambie el conjunto del Bloque 2 por círculos concéntricos y reentrene:

```python
from sklearn.datasets import make_circles
X2 = torch.tensor(make_circles(2000, noise=0.04, factor=0.5,
                               random_state=0)[0], dtype=torch.float32)
X2 = (X2 - X2.mean(0)) / X2.std(0)
```

Nada más cambia: ni el calendario, ni la red, ni los ciclos. Ésa es la
gracia del método.
